# Alpamayo Results Visualization

This notebook visualizes the output JSON from `run_on_openpilot.py`.
Ensure `llm-model-tests/alpamayo_results.json` exists.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# Path to results JSON
json_path = "../../llm-model-tests/alpamayo_results.json"

if not os.path.exists(json_path):
    # Fallback/Debug path if running locally or differently
    # Try relative to notebook just in case user moved it
    possible_paths = [
        "../../llm-model-tests/alpamayo_results.json",
        "../route_1_seg_00_results.json",
        "alpamayo_results.json"
    ]
    for p in possible_paths:
        if os.path.exists(p):
            json_path = p
            break

if not os.path.exists(json_path):
    print(f"Error: {json_path} not found.")
else:
    with open(json_path, 'r') as f:
        results = json.load(f)
    print(f"Loaded {len(results)} results from {json_path}.")

In [ ]:
def visualize_result(index):
    if index < 0 or index >= len(results):
        print("Index out of range")
        return

    entry = results[index]
    
    # Image Handling
    img_path = entry.get("image_path")
    
    # Adjust relative path if needed
    if img_path and img_path.startswith(".."):
        # Hacky fix: try adding another ../ if running from notebooks subdirectory
        notebook_rel_path = os.path.join("../", img_path)
        if os.path.exists(notebook_rel_path):
             img_path = notebook_rel_path
        elif os.path.exists(img_path):
             pass # Path is correct as is
    
    try:
        img = Image.open(img_path)
    except Exception as e:
        print(f"Could not load image at {img_path}: {e}")
        img = None

    path_pred = np.array(entry["pred_xyz"])
    path_gt = np.array(entry["gt_xyz"])
    reasoning = entry["reasoning"]

    # Plot
    plt.figure(figsize=(15, 6))

    # Subplot 1: Image
    plt.subplot(1, 2, 1)
    if img:
        plt.imshow(img)
    plt.title(f"Frame: {entry.get('filename', 'Unknown')}")
    plt.axis('off')

    # Subplot 2: Trajectory (BEV)
    plt.subplot(1, 2, 2)
    # Usually plotting requires swapping axes for map view or keeping as is.
    # Let's plot x vs y.
    plt.plot(path_pred[:, 1], path_pred[:, 0], 'b.-', label='Prediction')
    plt.plot(path_gt[:, 1], path_gt[:, 0], 'r.--', label='Ground Truth')
    plt.plot(0, 0, 'ko', markersize=10, label='Ego Start')
    plt.legend()
    plt.grid(True)
    plt.xlabel('Lateral (y) [m]')
    plt.ylabel('Longitudinal (x) [m]')
    plt.title("Trajectory Prediction (BEV)")
    plt.axis('equal')

    plt.tight_layout()
    plt.show()

    print("Reasoning Trace:")
    print("-" * 20)
    print(reasoning)
    print("-" * 20)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Output widget to hold the plot
out = widgets.Output()

# Limit slider to number of results
max_idx = len(results) - 1 if 'results' in globals() and results else 0

slider = widgets.IntSlider(min=0, max=max_idx, step=1, value=0, description='Frame:')
btn_prev = widgets.Button(description="Prev")
btn_next = widgets.Button(description="Next")

def show_current_frame(change=None):
    idx = slider.value
    with out:
        clear_output(wait=True)
        if 'results' in globals() and results:
             visualize_result(idx)
        else:
             print("No results loaded.")

def on_next(b):
    if slider.value < slider.max:
        slider.value += 1

def on_prev(b):
    if slider.value > slider.min:
        slider.value -= 1

# Link slider change to update function
slider.observe(show_current_frame, names='value')

# Link buttons to slider updates
btn_next.on_click(on_next)
btn_prev.on_click(on_prev)

# Initial display
display(widgets.HBox([btn_prev, slider, btn_next]))
display(out)
show_current_frame()